Eye Tracking Feature Extraction

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import logging
import importlib
import traceback

import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# P11, P12, P13 excluded — eye tracker did not work or gaze file missing
PARTICIPANTS = [f"P{i:02d}" for i in range(1, 21) if i not in (10, 11, 13)]

BASE_DIR  = "/content/drive/MyDrive/CAMES/data_collection_training"
PUPIL_COL = "pupil_smooth"

#Logging
importlib.reload(logging)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger(__name__)

In [ ]:
def assign_task_labels(events: pd.DataFrame, gaze: pd.DataFrame) -> pd.DataFrame:
      """Assign task labels to events based on their timestamps."""

    events = events.copy()
    task_gaze = gaze[gaze["task_label"].notna()].copy()
    task_windows = (
        task_gaze.groupby("task_label")["Timestamp Unix"]
        .agg(t_start="min", t_end="max")
        .reset_index()
    )
    device_origin = gaze["device_time_stamp"].min()
    unix_origin   = gaze["Timestamp Unix"].min()
    events["startT_unix"] = (events["startT"] * 1000 - device_origin) / 1e6 + unix_origin

    def find_task(t):
        """Return the task active at timestamp t."""
        match = task_windows[
            (task_windows["t_start"] <= t) &
            (task_windows["t_end"]   >= t)
        ]
        return match["task_label"].iloc[0] if len(match) else np.nan

    events["task_label"] = events["startT_unix"].apply(find_task)
    return events
def assign_exploration_flag(events: pd.DataFrame, gaze: pd.DataFrame) -> pd.DataFrame:
    """Mark events that occur during exploration periods."""
    events = events.copy()

    # Keep only gaze samples recorded during exploration.
    exp_gaze = gaze[gaze["in_exploration"] == True].copy()

    # If no exploration data exists, mark all events as False.
    if exp_gaze.empty:
        events["in_exploration"] = False
        return events

    # Split exploration samples into continuous intervals using 1 s gaps.
    exp_gaze = exp_gaze.sort_values("Timestamp Unix")
    exp_gaze["gap"] = exp_gaze["Timestamp Unix"].diff() > 1.0
    exp_gaze["interval_id"] = exp_gaze["gap"].cumsum()

    # Find the start and end time of each exploration interval.
    intervals = (
        exp_gaze.groupby("interval_id")["Timestamp Unix"]
        .agg(t_start="min", t_end="max")
        .reset_index(drop=True)
    )

    def in_exploration(t):
        """Return True if timestamp t falls inside an exploration interval."""
        return ((intervals["t_start"] <= t) & (intervals["t_end"] >= t)).any()

    # Mark each event based on its start timestamp.
    events["in_exploration"] = events["startT_unix"].apply(in_exploration)
    return events


def pupil_features(gaze: pd.DataFrame) -> list:
    """Calculate pupil and blink features for each task."""
    # Use only gaze samples recorded during exploration.
    gaze = gaze[gaze["in_exploration"] == True].copy()
    gaze = gaze.sort_values(["task_label", "time_s"])

    # Detect the start of each blink.
    gaze["blink"] = gaze["blink"].fillna(False).astype(bool)
    gaze["blink_onset"] = gaze.groupby("task_label")["blink"].transform(
        lambda s: s & ~s.shift(fill_value=False)
    )

    rows = []

    for task_label, g in gaze.groupby("task_label"):
        # Calculate total exploration duration for the task.
        dur_s = (
            g.groupby("exploration_id")["time_s"]
            .apply(lambda x: x.max() - x.min())
            .sum()
        )

        p = g[PUPIL_COL]

        # Find the first valid pupil measurement.
        valid_pupil = g.loc[
            g[PUPIL_COL].notna(),
            ["time_s", PUPIL_COL]
        ]
        first_valid_t = (
            valid_pupil["time_s"].min()
            if not valid_pupil.empty
            else np.nan
        )

        # Use the first 3 seconds of valid pupil data as baseline.
        if not np.isnan(first_valid_t):
            base_mask = (
                (g["time_s"] >= first_valid_t) &
                (g["time_s"] < first_valid_t + 3)
            )
            baseline = g.loc[base_mask, PUPIL_COL].mean()
        else:
            baseline = np.nan

        # Calculate baseline-corrected mean pupil size.
        pupil_mean = (p - baseline).mean()

        # Estimate pupil-size change over time.
        m = g["time_s"].notna() & p.notna()
        pupil_slope = (
            np.polyfit(
                g.loc[m, "time_s"].to_numpy(),
                p.loc[m].to_numpy(),
                1
            )[0]
            if m.sum() > 10 else np.nan
        )

        # Calculate blink-related measures.
        blink_time_frac = g["blink"].mean()
        blink_per_min = (
            g["blink_onset"].sum() / (dur_s / 60)
            if dur_s > 0 else np.nan
        )

        # Store the calculated task-level features.
        rows.append({
            "task_label": task_label,
            "task_duration_s": round(dur_s, 2),
            "pupil_mean_bc": round(float(pupil_mean), 4)
            if not np.isnan(float(pupil_mean)) else np.nan,
            "pupil_slope": round(float(pupil_slope), 8)
            if not np.isnan(float(pupil_slope)) else np.nan,
            "blink_time_frac": round(float(blink_time_frac), 4),
            "blink_per_min": round(float(blink_per_min), 2)
            if not np.isnan(float(blink_per_min)) else np.nan,
            "pupil_valid_pct": round(p.notna().mean() * 100, 1),
        })

    return rows


def fixation_features(fix: pd.DataFrame) -> list:
    """Calculate fixation features for each task."""
    # Keep only fixations occurring during exploration.
    fix = fix[fix["in_exploration"] == True].copy()

    rows = []

    for task_label, grp in fix.groupby("task_label"):
        # Convert total fixation duration from milliseconds to seconds.
        dur_s = grp["dur"].sum() / 1000

        # Store fixation count, rate, and duration statistics.
        rows.append({
            "task_label": task_label,
            "fix_count": len(grp),
            "fix_rate_hz": round(len(grp) / dur_s, 3)
            if dur_s > 0 else np.nan,
            "fix_dur_mean_s": round(grp["dur"].mean() / 1000, 4),
            "fix_dur_std_s": round(grp["dur"].std() / 1000, 4),
            "fix_dur_total_s": round(grp["dur"].sum() / 1000, 2),
        })

    return rows


def saccade_features(sac: pd.DataFrame) -> list:
    """Calculate saccade features for each task."""
    # Keep only valid saccades occurring during exploration.
    sac = sac[
        (sac["sclass"] == "S") &
        (sac["in_exploration"] == True)
    ].copy()

    rows = []

    for task_label, grp in sac.groupby("task_label"):
        # Convert total saccade duration from milliseconds to seconds.
        dur_s = grp["dur"].sum() / 1000

        # Store saccade count, rate, and amplitude statistics.
        rows.append({
            "task_label": task_label,
            "sac_count": len(grp),
            "sac_rate_hz": round(len(grp) / dur_s, 3)
            if dur_s > 0 else np.nan,
            "sac_amp_mean_px": round(grp["amp"].mean(), 4),
            "sac_amp_std_px": round(grp["amp"].std(), 4),
        })

    return rows

In [ ]:
def main():
    """Extract, combine, standardize, and save eye-tracking features.

    Processes each participant's gaze, fixation, and saccade data, assigns
    task and exploration labels, extracts task-level eye-tracking features,
    computes participant-wise z-scores, and saves the combined dataset to CSV.


    """
    log.info("Processing %d participant(s).", len(PARTICIPANTS))
    all_features = []

    for pid in PARTICIPANTS:
        print(f"Extracting {pid}...")
        log.info("── %s ──────────────────────────────────────────", pid)

        eye_dir = Path(BASE_DIR) / pid / f"{pid}_session1" / f"eye_{pid}"

        try:
            gaze = pd.read_csv(eye_dir / "gaze_with_pupil.csv", low_memory=False)
            fix  = pd.read_csv(eye_dir / "fixations.csv",       low_memory=False)
            sac  = pd.read_csv(eye_dir / "saccades.csv",        low_memory=False)
        except FileNotFoundError as e:
            log.error("  Missing file — %s", e)
            continue

        try:
            # Ensure device_time_stamp is numeric in gaze
            gaze["device_time_stamp"] = pd.to_numeric(gaze["device_time_stamp"], errors="coerce")

            # Assign task labels and exploration flags to fixations and saccades
            fix = assign_task_labels(fix, gaze)
            sac = assign_task_labels(sac, gaze)
            fix = assign_exploration_flag(fix, gaze)
            sac = assign_exploration_flag(sac, gaze)

            # Keep only events within tasks
            fix = fix[fix["task_label"].notna()]
            sac = sac[sac["task_label"].notna()]

            log.info(
                "  Fix in exploration: %d / %d   Sac in exploration: %d / %d",
                fix["in_exploration"].sum(), len(fix),
                sac["in_exploration"].sum(), len(sac),
            )

            # Extract features
            pup_df = pd.DataFrame(pupil_features(gaze))
            fix_df = pd.DataFrame(fixation_features(fix))
            sac_df = pd.DataFrame(saccade_features(sac))

            # Merge on task_label
            merged = (
                pup_df
                .merge(fix_df, on="task_label", how="outer")
                .merge(sac_df, on="task_label", how="outer")
            )
            merged.insert(0, "participant", pid)

            all_features.append(merged)
            log.info("  Done — %d tasks extracted", len(merged))

        except Exception:
            log.error("  FAILED:\n%s", traceback.format_exc())

    #Combine
    if not all_features:
        log.error("No features extracted — check errors above.")
        return

    eye_features = (
        pd.concat(all_features, ignore_index=True)
        .sort_values(["participant", "task_label"])
        .reset_index(drop=True)
    )

    #Z-score eye features per participant
    eye_feature_cols = [
        'pupil_mean_bc', 'pupil_slope', 'blink_time_frac', 'blink_per_min',
        'pupil_valid_pct', 'fix_count', 'fix_rate_hz', 'fix_dur_mean_s',
        'fix_dur_std_s', 'fix_dur_total_s', 'sac_count', 'sac_rate_hz',
        'sac_amp_mean_px', 'sac_amp_std_px'
    ]

    for col in eye_feature_cols:
        z_col = f"{col}_z"
        eye_features[z_col] = eye_features.groupby("participant")[col].transform(
            lambda x: (x - x.mean()) / x.std(ddof=1) if x.std(ddof=1) > 0 else 0.0
        )

    #Save
    out_path = os.path.join(BASE_DIR, "eye_features_all_participants.csv")
    eye_features.to_csv(out_path, index=False)
    log.info("Saved → %s", out_path)
    log.info("Shape: %s", eye_features.shape)
    print(eye_features.to_string())


main()

11:17:22  INFO      Processing 17 participant(s).
11:17:22  INFO      ── P01 ──────────────────────────────────────────


Extracting P01...


11:17:51  INFO        Fix in exploration: 3234 / 7322   Sac in exploration: 3230 / 7320
11:17:51  INFO        Done — 10 tasks extracted
11:17:51  INFO      ── P02 ──────────────────────────────────────────


Extracting P02...


11:18:22  INFO        Fix in exploration: 3995 / 6641   Sac in exploration: 3994 / 6637
11:18:22  INFO        Done — 7 tasks extracted
11:18:22  INFO      ── P03 ──────────────────────────────────────────


...

Extracting P19...


11:26:25  INFO        Fix in exploration: 5226 / 8299   Sac in exploration: 5227 / 8298
11:26:25  INFO        Done — 9 tasks extracted
11:26:25  INFO      ── P20 ──────────────────────────────────────────


Extracting P20...


11:27:07  INFO        Fix in exploration: 5694 / 11313   Sac in exploration: 5693 / 11316
11:27:08  INFO        Done — 10 tasks extracted
11:27:08  INFO      Saved → /content/drive/MyDrive/CAMES/data_collection_training/eye_features_all_participants.csv
11:27:08  INFO      Shape: (152, 31)


    participant       task_label  task_duration_s  pupil_mean_bc  pupil_slope  blink_time_frac  blink_per_min  pupil_valid_pct  fix_count  fix_rate_hz  fix_dur_mean_s  fix_dur_std_s  fix_dur_total_s  sac_count  sac_rate_hz  sac_amp_mean_px  sac_amp_std_px  pupil_mean_bc_z  pupil_slope_z  blink_time_frac_z  blink_per_min_z  pupil_valid_pct_z  fix_count_z  fix_rate_hz_z  fix_dur_mean_s_z  fix_dur_std_s_z  fix_dur_total_s_z  sac_count_z  sac_rate_hz_z  sac_amp_mean_px_z  sac_amp_std_px_z
0           P01        task1_5_2           281.57         0.2554     0.001734           0.2168          40.91             96.2        665        2.784          0.3592         0.2943           238.88        498       40.738         120.3828        147.4916        -0.523753      -0.816114           0.026640         0.506610           0.356486     1.795888      -1.091335          1.312584         1.229281           1.858610     1.845856       0.276935          -0.200534         -0.330825
1           P01     